# 01: Environment, Data and Splits

Thesis: *Convolutional Neural Networks in Knee MRI Imaging*, John Sciuto (24814764), UTS Honours Project.

This notebook sets up everything the training notebook needs, including:

1. checks the Python environment and the NVIDIA GPU backend
2. checks the MRNet dataset at `data/MRNet-v1.0/`
3. builds the development/tuning split from thesis section 3.3 and the numbers for Table 3.2
4. saves the dataset figures for Chapter 3
5. (optional, section 1.3) prepares KneeMRI for external validation (discussed in Future Work)

`SMOKE_TEST = True` switches the whole pipeline to a small synthetic
dataset. Note this is only used for testing the pipeline/code changes (files separately go to separate `outputs_smoke/`
and `splits_smoke/` folders, so as to avoid being mixed with the real results :))

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))   # make src/ importable from notebooks/

import numpy as np
import pandas as pd
import torch

from src.config import Config, TASKS, PLANES, get_device, env_report
from src import data as D
from src import plots as P

P.setup_style()
print(env_report())          # paste this block into thesis §3.11
print("\nDevice selected:", get_device())

## 1.1 Configuration

Expected MRNet layout at `data/MRNet-v1.0/` (already in place):

```
train/{axial,coronal,sagittal}/*.npy   valid/{axial,coronal,sagittal}/*.npy
{train,valid}-{abnormal,acl,meniscus}.csv
```

In [ ]:
SMOKE_TEST = False   # for testing pipeline

cfg = Config(smoke_test=SMOKE_TEST).make_dirs()

# cfg.mrnet_dir = Path("/path/to/MRNet-v1.0")

if SMOKE_TEST:
    D.make_smoke_dataset(cfg.smoke_dir)
    print("Smoke dataset ready at", cfg.smoke_dir)

assert (cfg.data_root / "train-acl.csv").exists(), f"MRNet not found at {cfg.data_root}"
n_train = len(list((cfg.data_root / "train" / "sagittal").glob("*.npy")))
n_valid = len(list((cfg.data_root / "valid" / "sagittal").glob("*.npy")))
print(f"MRNet found at {cfg.data_root}: {n_train} train / {n_valid} valid exams per plane")
cfg.save()

## 1.2 Labels, prevalence and the development/tuning split

The 1,130 public training exams get split into a development set and a tuning set using
iterative multi-label stratification with a fixed seed, which keeps the joint distribution of
the three labels similar in both sets (section 3.3). The public `valid/` set (120 exams) is the
held-out test set. Nothing touches its images until the final evaluation in notebook 03.

The printed table is Table 3.2 of the thesis (also saved to `outputs/tables/`).

In [ ]:
labels_train = D.load_mrnet_labels(cfg.data_root, "train")
labels_valid = D.load_mrnet_labels(cfg.data_root, "valid")

n_tune = cfg.n_tuning if not SMOKE_TEST else max(4, len(labels_train) // 5)
dev_ids, tune_ids = D.make_dev_tuning_split(labels_train, n_tuning=n_tune, seed=0)
pd.DataFrame({"id": dev_ids}).to_csv(cfg.splits_dir / "dev_ids.csv", index=False)
pd.DataFrame({"id": tune_ids}).to_csv(cfg.splits_dir / "tune_ids.csv", index=False)
print(f"development {len(dev_ids)} · tuning {len(tune_ids)} · test {len(labels_valid)}")

table_3_2 = D.prevalence_table({
    "Development training": labels_train[labels_train.id.isin(dev_ids)],
    "Tuning": labels_train[labels_train.id.isin(tune_ids)],
    "Test (public MRNet validation)": labels_valid,
})
table_3_2.to_csv(cfg.tables_dir / "table_3_2_partitions.csv", index=False)
display(table_3_2)

### Dataset figures for Chapter 3

A slice montage of one ACL-positive exam in each plane (useful as a Figure 3.x to show what the
model actually sees), plus the slices-per-exam histogram that justifies one-exam batches
(section 3.6).

In [ ]:
eid = labels_train.loc[labels_train.acl == 1, "id"].iloc[0]
for plane in PLANES:
    vol = np.load(cfg.data_root / "train" / plane / f"{eid}.npy")
    P.exam_montage(vol, f"MRNet exam {eid} — {plane} ({vol.shape[0]} slices)",
                   save=cfg.figures_dir / f"dataset_montage_{plane}.png")

sample = labels_train.id.iloc[:200]   # header-only reads via mmap, so this is quick
counts = {p: [np.load(cfg.data_root / "train" / p / f"{i}.npy", mmap_mode="r").shape[0]
              for i in sample] for p in PLANES}
P.slice_count_hist(counts, save=cfg.figures_dir / "slice_count_hist.png")

## 1.3 Optional: KneeMRI preparation for external validation (section 3.9)

Skip this section for training. It only matters if the KneeMRI dataset gets downloaded later to
fill thesis section 4.7 (zero-shot external ACL validation, SQ4). If `data/kneemri/` exists
(`metadata.csv` plus `volumetric_data/`), these cells convert each 12-bit volume once to an
MRNet-style uint8 array, save the binary ACL labels (partial + complete rupture = injured,
following Bien et al., 2018), and draw the radiologist ROI figure. Only load pickle files from
the official Zenodo record (record 14789903), since pickle files can run arbitrary code.

In [ ]:
HAVE_KNEEMRI = (cfg.kneemri_dir / "metadata.csv").exists()
if HAVE_KNEEMRI:
    meta, cols = D.load_kneemri_meta(cfg.kneemri_dir)
    print("Columns detected:", cols)
    display(meta[cols["diag"]].value_counts().rename("exams per diagnosis code"))

    n_new = D.convert_kneemri(cfg.kneemri_dir, cfg.kneemri_npy_dir, meta, cols["file"])
    print(f"Converted {n_new} new volumes -> {cfg.kneemri_npy_dir}")

    meta[["id", "acl"]].to_csv(cfg.splits_dir / "kneemri_labels.csv", index=False)
    print(f"Labels saved: {int(meta.acl.sum())} injured / {len(meta)} total")
else:
    print("KneeMRI not present — external validation (§4.7) will be skipped in notebook 03.")

In [ ]:
# Radiologist-annotated ACL region on its central slice (figure for §3.2.2; the
# ground-truth counterpart of the Grad-CAM detection boxes in §4.8).
if HAVE_KNEEMRI and all(cols[k] for k in ("roi_x", "roi_y", "roi_z", "roi_w", "roi_h", "roi_d")):
    row = meta[meta.acl == 1].iloc[0]
    vol = D.load_pck(cfg.kneemri_dir / "volumetric_data" / str(row[cols["file"]]))
    k = int(row[cols["roi_z"]] + row[cols["roi_d"]] // 2)
    box = (row[cols["roi_x"]], row[cols["roi_y"]], row[cols["roi_w"]], row[cols["roi_h"]])
    P.roi_example_figure(vol / vol.max(), k, box, row["id"],
                         save=cfg.figures_dir / "kneemri_roi_example.png")

In [ ]:
# Smoke mode only: fabricate a tiny synthetic "external" set so notebook 03's
# external-validation cell can be exercised without KneeMRI.
if SMOKE_TEST and not HAVE_KNEEMRI:
    rng = np.random.default_rng(1)
    cfg.kneemri_npy_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for i in range(20):
        eid, y = f"ext{i:03d}", i % 2
        s = int(rng.integers(18, 30))
        vol = rng.normal(90, 25, (s, 256, 256))
        if y:
            zz, yy, xx = np.ogrid[:s, :256, :256]
            vol[((zz - s // 2) / 4) ** 2 + ((yy - 128) / 28) ** 2 + ((xx - 128) / 28) ** 2 < 1] += 150
        np.save(cfg.kneemri_npy_dir / f"{eid}.npy", np.clip(vol, 0, 255).astype(np.uint8))
        rows.append({"id": eid, "acl": float(y)})
    pd.DataFrame(rows).to_csv(cfg.splits_dir / "kneemri_labels.csv", index=False)
    print("Synthetic external set written (smoke mode).")

## Done

This created the split files in `splits/`, Table 3.2 in `outputs/tables/` and the dataset
figures in `outputs/figures/`. Next notebook: `02_training.ipynb`.